# 위니쿠키 RAG - ChromaDB 임베딩 파이프라인

Knowledge Base(winicookie_knowledge.json)를 bge-m3 임베딩 모델을 사용하여 ChromaDB에 저장하는 파이프라인입니다.

## 1. 라이브러리 임포트 및 데이터 로드

In [38]:
import json
import chromadb
import requests

# Knowledge Base 로드
with open('winicookie_knowledge.json', 'r', encoding='utf-8') as f:
    knowledge = json.load(f)

print(f'총 문서 수: {len(knowledge)}개')
print(f'첫 번째 문서 예시:')
knowledge[0]

총 문서 수: 46개
첫 번째 문서 예시:


{'id': 'store_name', 'category': '가게정보', 'content': '위니쿠키는 수제 쿠키 전문점입니다.'}

## 2. Ollama bge-m3 임베딩 함수 정의

ChromaDB는 커스텀 임베딩 함수를 지원합니다.  
Ollama의 bge-m3를 호출하는 함수를 만들어 ChromaDB에 연결합니다.

### 핵심 구조
- ChromaDB의 `EmbeddingFunction` 인터페이스를 상속
- `__call__` 메서드에서 Ollama API를 호출하여 텍스트 → 벡터 변환
- Ollama는 로컬에서 실행되므로 `http://localhost:11434` 로 요청

In [39]:
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

class OllamaEmbeddingFunction(EmbeddingFunction):
    """
    Ollama 로컬 서버의 임베딩 모델을 ChromaDB에서 사용하기 위한 커스텀 함수
    
    - model: Ollama에 설치된 임베딩 모델 이름
    - url: Ollama 서버 주소 (기본: localhost:11434)
    """
    def __init__(self, model: str = 'bge-m3', url: str = 'http://localhost:11434'):
        self.model = model
        self.url = url
    
    def __call__(self, input: Documents) -> Embeddings:
        """
        텍스트 리스트를 받아 임베딩 벡터 리스트를 반환
        
        input: ['텍스트1', '텍스트2', ...]
        return: [[0.1, 0.2, ...], [0.3, 0.4, ...], ...]
        """
        embeddings = []
        for text in input:
            response = requests.post(
                f'{self.url}/api/embed',
                json={'model': self.model, 'input': text}
            )
            embeddings.append(response.json()['embeddings'][0])
        return embeddings

# 임베딩 함수 생성 및 테스트
ollama_ef = OllamaEmbeddingFunction(model='bge-m3')

# 테스트: 한국어 문장 임베딩
test_embedding = ollama_ef(['위니쿠키 테스트'])
print(f'임베딩 벡터 차원: {len(test_embedding[0])}')
print(f'벡터 앞 5개 값: {test_embedding[0][:5]}')

임베딩 벡터 차원: 1024
벡터 앞 5개 값: [-0.03195062  0.0394392  -0.01920219 -0.00610602 -0.0154315 ]


## 3. ChromaDB 컬렉션 생성

### 주요 설정
- `PersistentClient`: 데이터를 디스크에 저장하여 프로그램 종료 후에도 유지
- `get_or_create_collection`: 이미 존재하면 기존 컬렉션 사용, 없으면 새로 생성
- `embedding_function`: bge-m3 임베딩 함수를 연결
- `metadata`: 유사도 측정 방식을 cosine으로 설정 (임베딩 모델과 가장 일반적인 조합)

In [40]:
# PersistentClient: 디스크에 저장 (프로그램 종료 후에도 데이터 유지)
client = chromadb.PersistentClient(path='./chroma_db')

# 컬렉션 생성 (이미 있으면 기존 것 사용)
collection = client.get_or_create_collection(
    name='winicookie_knowledge',
    embedding_function=ollama_ef,
    metadata={'hnsw:space': 'cosine'}  # 코사인 유사도 사용
)

print(f'컬렉션 이름: {collection.name}')
print(f'현재 저장된 문서 수: {collection.count()}')

컬렉션 이름: winicookie_knowledge
현재 저장된 문서 수: 46


## 4. Knowledge Base → ChromaDB 저장

### collection.add() 파라미터 매핑
- `ids` ← knowledge의 `id` 필드 (고유 식별자)
- `documents` ← knowledge의 `content` 필드 (원본 텍스트, 자동 임베딩됨)
- `metadatas` ← knowledge의 `category` 필드 (필터 검색용 메타데이터)

In [41]:
# 기존 데이터가 있으면 초기화 (재실행 시 중복 방지)
if collection.count() > 0:
    # 기존 컬렉션 삭제 후 재생성
    client.delete_collection('winicookie_knowledge')
    collection = client.get_or_create_collection(
        name='winicookie_knowledge',
        embedding_function=ollama_ef,
        metadata={'hnsw:space': 'cosine'}
    )
    print('기존 컬렉션 초기화 완료')

# Knowledge Base 데이터 추출
ids = [item['id'] for item in knowledge]
documents = [item['content'] for item in knowledge]
metadatas = [{'category': item['category']} for item in knowledge]

# ChromaDB에 저장 (bge-m3로 자동 임베딩)
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f'저장 완료! 총 {collection.count()}개 문서가 ChromaDB에 저장되었습니다.')

기존 컬렉션 초기화 완료
저장 완료! 총 46개 문서가 ChromaDB에 저장되었습니다.


## 5. 저장 확인 및 검색 테스트

실제 사용자 질문을 시뮬레이션하여 검색이 잘 되는지 확인합니다.

In [42]:
# 테스트 1: 일반 검색 - 가격 질문
results = collection.query(
    query_texts=['초코퍼지 얼마예요?'],
    n_results=3
)

print('=== 테스트 1: "초코퍼지 얼마예요?" ===')
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0], 
    results['distances'][0]
)):
    print(f'{i+1}. [{meta["category"]}] {doc}')
    print(f'   유사도 거리: {dist:.4f}')
    print()

=== 테스트 1: "초코퍼지 얼마예요?" ===
1. [메뉴] 초코퍼지 가격은 3,500원입니다.
   유사도 거리: 0.1766

2. [메뉴] 크랜베리&화이트초코칩 가격은 2,600원입니다.
   유사도 거리: 0.4493

3. [메뉴] 오렌지쿠키 가격은 2,600원입니다.
   유사도 거리: 0.4930



In [43]:
# 테스트 2: 카테고리 필터 검색 - 영업시간
results = collection.query(
    query_texts=['몇 시에 문 닫아요?'],
    where={'category': '영업시간'},
    n_results=3
)

print('=== 테스트 2: "몇 시에 문 닫아요?" (영업시간 필터) ===')
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0], 
    results['distances'][0]
)):
    print(f'{i+1}. [{meta["category"]}] {doc}')
    print(f'   유사도 거리: {dist:.4f}')
    print()

=== 테스트 2: "몇 시에 문 닫아요?" (영업시간 필터) ===
1. [영업시간] 위니쿠키는 오후 5시에 마감합니다. 저녁이나 야간에는 운영하지 않습니다.
   유사도 거리: 0.3322

2. [영업시간] 위니쿠키 영업시간은 월요일부터 토요일까지 오전 10시 30분부터 오후 5시까지입니다.
   유사도 거리: 0.4278

3. [영업시간] 일요일에는 위니쿠키가 운영하지 않습니다. 일요일은 정기휴무이며 월요일부터 토요일까지만 영업합니다.
   유사도 거리: 0.4898



In [44]:
# 테스트 3: 위치 관련 질문
results = collection.query(
    query_texts=['어떻게 찾아가요?'],
    n_results=3
)

print('=== 테스트 3: "어떻게 찾아가요?" ===')
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0], 
    results['distances'][0]
)):
    print(f'{i+1}. [{meta["category"]}] {doc}')
    print(f'   유사도 거리: {dist:.4f}')
    print()

=== 테스트 3: "어떻게 찾아가요?" ===
1. [위치] 충청대로를 따라 오시다 파크랜드 간판이 보이면 속도를 줄이고, 파크랜드 지나 하우트커피베이커리 앞으로 들어오시면 됩니다.
   유사도 거리: 0.4891

2. [주문] 결제 방법은 매장에서 확인하거나 010-3387-5313으로 문의하시면 안내받을 수 있습니다.
   유사도 거리: 0.5022

3. [영업시간] 재료 소진 시 조기 마감될 수 있으므로 방문 전 전화로 확인하는 것이 좋습니다.
   유사도 거리: 0.5234



In [45]:
# 테스트 4: 환각 방지 테스트 - 존재하지 않는 메뉴 질문
results = collection.query(
    query_texts=['마카롱 있어요?'],
    n_results=3
)

print('=== 테스트 4: "마카롱 있어요?" (존재하지 않는 메뉴) ===')
print('※ 마카롱은 메뉴에 없으므로, 검색 결과의 유사도 거리가 높을수록 정상')
print()
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0], 
    results['distances'][0]
)):
    print(f'{i+1}. [{meta["category"]}] {doc}')
    print(f'   유사도 거리: {dist:.4f}')
    print()

=== 테스트 4: "마카롱 있어요?" (존재하지 않는 메뉴) ===
※ 마카롱은 메뉴에 없으므로, 검색 결과의 유사도 거리가 높을수록 정상

1. [위치] 위니쿠키 앞쪽에 공용주차장이 있습니다.
   유사도 거리: 0.4922

2. [주문] 당일 준비된 쿠키는 매장 상황에 따라 달라질 수 있습니다. 방문 전 010-3387-5313으로 문의하시면 확인 가능합니다.
   유사도 거리: 0.4926

3. [메뉴] 위니쿠키는 쿠키 전문점으로 쿠키와 휘낭시에만 판매합니다. 케이크, 마카롱, 빵, 음료 등은 판매하지 않습니다.
   유사도 거리: 0.5109



## 6. 전체 저장 데이터 확인

In [46]:
# 저장된 전체 데이터 조회
all_data = collection.get(include=['documents', 'metadatas'])

print(f'총 저장 문서: {len(all_data["ids"])}개\n')

# 카테고리별 정리
from collections import defaultdict
by_category = defaultdict(list)
for id_, doc, meta in zip(all_data['ids'], all_data['documents'], all_data['metadatas']):
    by_category[meta['category']].append((id_, doc))

for category, items in sorted(by_category.items()):
    print(f'\n[{category}] - {len(items)}개')
    print('-' * 50)
    for id_, doc in items:
        print(f'  {id_}: {doc}')

총 저장 문서: 46개


[가게정보] - 3개
--------------------------------------------------
  store_name: 위니쿠키는 수제 쿠키 전문점입니다.
  store_contact: 위니쿠키 연락처는 010-3387-5313입니다.
  store_single_branch: 위니쿠키는 충북 청주시 청원구 내수읍에 본점 한 곳만 운영하고 있습니다. 강남점, 홍대점 등 다른 지점은 없습니다.

[메뉴] - 21개
--------------------------------------------------
  menu_list: 위니쿠키 메뉴는 비스퀴, 르뱅쿠키, 초코퍼지, 아망디오쇼콜라, 카야잼쿠키, 오렌지쿠키, 사브레쿠키, 크랜베리&화이트초코칩, 땅콩버터쿠키, 딸기잼쿠키, 헤이!누텔라쿠키, 헤이즐넛둘세쿠키, 메이플피칸쿠키, 하트버터쿠키, 휘낭시에, 버터쿠키 총 16종입니다.
  menu_bisqui: 비스퀴 가격은 3,700원입니다.
  menu_levain: 르뱅쿠키 가격은 4,300원입니다.
  menu_choco_fudge: 초코퍼지 가격은 3,500원입니다.
  menu_amandio: 아망디오쇼콜라 가격은 2,600원입니다.
  menu_kaya_jam: 카야잼쿠키 가격은 2,900원입니다.
  menu_orange: 오렌지쿠키 가격은 2,600원입니다.
  menu_sable: 사브레쿠키 가격은 2,200원입니다.
  menu_cranberry_whitechoco: 크랜베리&화이트초코칩 가격은 2,600원입니다.
  menu_peanut_butter: 땅콩버터쿠키 가격은 2,300원입니다.
  menu_strawberry_jam: 딸기잼쿠키 가격은 2,900원입니다.
  menu_nutella: 헤이!누텔라쿠키 가격은 2,900원입니다.
  menu_hazelnut_dulce: 헤이즐넛둘세쿠키 가격은 2,900원입니다.
  menu_maple_pecan: 메이플피칸쿠키 가격은 2,600원입니다.
  menu

---

# 3단계: RAG 파이프라인 개발

ChromaDB 검색 결과를 활용하여 Bllossom-3B가 정확한 답변을 생성하는 RAG(Retrieval-Augmented Generation) 파이프라인입니다.

### 파이프라인 흐름
```
사용자 질문 → ChromaDB 검색 → 임계값 판단 → 프롬프트 조합 → Bllossom-3B 답변 생성
```

### 환각 방지 이중 안전장치
- **1차 (거친 필터)**: 유사도 거리 임계값으로 관련 없는 질문 사전 차단
- **2차 (정밀 필터)**: 프롬프트 지시로 참고 정보에 없는 내용 답변 방지

## 모듈 1: Knowledge Base 검색 함수

사용자 질문을 받아 ChromaDB에서 관련 문서를 검색하고,  
유사도 거리(distance)와 함께 결과를 반환합니다.

- `n_results=3`: 복합 질문(예: 가격+칼로리) 대응을 위해 3개 검색
- 반환값: 문서 내용, 카테고리, 유사도 거리를 묶은 리스트

In [47]:
def search_knowledge(query: str, n_results: int = 3) -> list:
    """
    사용자 질문으로 ChromaDB에서 관련 문서를 검색
    
    Args:
        query: 사용자 질문 텍스트
        n_results: 검색할 문서 수 (기본값: 3)
    
    Returns:
        list of dict: [{'content': str, 'category': str, 'distance': float}, ...]
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    # 검색 결과를 보기 쉬운 딕셔너리 리스트로 변환
    search_results = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        search_results.append({
            'content': doc,
            'category': meta['category'],
            'distance': dist
        })
    
    return search_results

# 함수 테스트
test = search_knowledge('초코퍼지 가격')
for r in test:
    print(f"[{r['category']}] {r['content']}")
    print(f"  거리: {r['distance']:.4f}")

[메뉴] 초코퍼지 가격은 3,500원입니다.
  거리: 0.1453
[메뉴] 크랜베리&화이트초코칩 가격은 2,600원입니다.
  거리: 0.4232
[메뉴] 오렌지쿠키 가격은 2,600원입니다.
  거리: 0.4605


## 모듈 2: 임계값 판단 (환각 방지 1차 필터)

검색 결과의 유사도 거리를 기준으로, 관련 정보가 있는지 판단합니다.

### 임계값 기준 (2단계 테스트 결과 기반)
- 정답이 있는 질문 (초코퍼지 가격): 거리 ~0.17
- 정답이 없는 질문 (마카롱): 거리 ~0.49~0.53
- **임계값 0.450**: 두 그룹 사이의 안전한 경계선

### 판단 로직
- 최소 거리 ≤ 0.450 → 관련 문서 존재 → 답변 생성 진행
- 최소 거리 > 0.450 → 관련 문서 없음 → "해당 정보가 없습니다" 응답

In [48]:
# 임계값 설정 (테스트 결과 기반)
DISTANCE_THRESHOLD = 0.45

def filter_by_threshold(search_results: list, threshold: float = DISTANCE_THRESHOLD) -> dict:
    """
    검색 결과의 유사도 거리를 기준으로 관련 정보 존재 여부를 판단
    
    Args:
        search_results: search_knowledge()의 반환값
        threshold: 유사도 거리 임계값 (기본값: 0.40)
    
    Returns:
        dict: {
            'has_answer': bool (관련 정보 존재 여부),
            'filtered_results': list (임계값 통과한 문서만),
            'min_distance': float (가장 가까운 거리)
        }
    """
    # 임계값 이하인 결과만 필터링
    filtered = [r for r in search_results if r['distance'] <= threshold]
    min_distance = min(r['distance'] for r in search_results)
    
    return {
        'has_answer': len(filtered) > 0,
        'filtered_results': filtered,
        'min_distance': min_distance
    }

# 테스트: 정답이 있는 질문
print('=== 정답이 있는 질문 ===')
results = search_knowledge('초코퍼지 가격')
filtered = filter_by_threshold(results)
print(f"관련 정보 존재: {filtered['has_answer']}")
print(f"최소 거리: {filtered['min_distance']:.4f}")
print(f"통과 문서 수: {len(filtered['filtered_results'])}개")

print()

# 테스트: 정답이 없는 질문
print('=== 정답이 없는 질문 ===')
results = search_knowledge('마카롱 있어요?')
filtered = filter_by_threshold(results)
print(f"관련 정보 존재: {filtered['has_answer']}")
print(f"최소 거리: {filtered['min_distance']:.4f}")
print(f"통과 문서 수: {len(filtered['filtered_results'])}개")

=== 정답이 있는 질문 ===
관련 정보 존재: True
최소 거리: 0.1453
통과 문서 수: 2개

=== 정답이 없는 질문 ===
관련 정보 존재: False
최소 거리: 0.4922
통과 문서 수: 0개


## 모듈 3: 프롬프트 조합 (환각 방지 2차 필터)

검색된 문서와 사용자 질문을 결합하여 Bllossom-3B에 전달할 프롬프트를 생성합니다.

### 프롬프트 설계 핵심
- **시스템 지시**: 참고 정보만 근거로 답변하도록 명시적으로 제한
- **참고 정보**: ChromaDB에서 검색된 문서 삽입
- **사용자 질문**: 원래 질문 그대로 전달

이 프롬프트 지시가 임계값을 아슬아슬하게 통과한 경우의 환각을 방지하는 2차 안전장치입니다.

In [49]:
def build_prompt(query: str, filtered_results: list) -> str:
    """
    검색 결과와 사용자 질문을 결합하여 LLM 프롬프트를 생성
    
    Args:
        query: 사용자 질문
        filtered_results: 임계값을 통과한 검색 결과 리스트
    
    Returns:
        str: Bllossom-3B에 전달할 완성된 프롬프트
    """
    # 참고 정보 텍스트 생성
    context_lines = []
    for i, r in enumerate(filtered_results, 1):
        context_lines.append(f"{i}. [{r['category']}] {r['content']}")
    context_text = '\n'.join(context_lines)
    
    # 프롬프트 템플릿
    prompt = f"""너는 '위니쿠키' 쿠키 전문점의 친절한 안내 챗봇이야.
아래 [참고 정보]만을 근거로 고객의 질문에 답변해.

중요한 규칙:
- 참고 정보에 있는 내용만 답변할 것
- 참고 정보에 없는 내용은 절대 추측하거나 만들어내지 말 것
- 참고 정보에 없으면 "죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요."라고 답할 것
- 짧고 친절하게 답변할 것

[참고 정보]
{context_text}

[고객 질문]
{query}

[답변]"""
    
    return prompt

# 프롬프트 생성 테스트
results = search_knowledge('초코퍼지 가격')
filtered = filter_by_threshold(results)
prompt = build_prompt('초코퍼지 가격', filtered['filtered_results'])

print('=== 생성된 프롬프트 ===')
print(prompt)

=== 생성된 프롬프트 ===
너는 '위니쿠키' 쿠키 전문점의 친절한 안내 챗봇이야.
아래 [참고 정보]만을 근거로 고객의 질문에 답변해.

중요한 규칙:
- 참고 정보에 있는 내용만 답변할 것
- 참고 정보에 없는 내용은 절대 추측하거나 만들어내지 말 것
- 참고 정보에 없으면 "죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요."라고 답할 것
- 짧고 친절하게 답변할 것

[참고 정보]
1. [메뉴] 초코퍼지 가격은 3,500원입니다.
2. [메뉴] 크랜베리&화이트초코칩 가격은 2,600원입니다.

[고객 질문]
초코퍼지 가격

[답변]


## 모듈 4: Bllossom-3B 답변 생성

Ollama API를 통해 파인튜닝된 Bllossom-3B 모델로 답변을 생성합니다.

### 생성 파라미터
- `temperature: 0.3`: 낮은 값으로 설정하여 사실 기반의 일관된 답변 유도 (창의성 ↓, 정확성 ↑)
- `top_p: 0.9`: 상위 90% 확률의 토큰만 고려
- `num_predict: 256`: 최대 생성 토큰 수 제한 (불필요한 장문 방지)

In [50]:
def generate_answer(prompt: str) -> str:
    """
    Ollama API를 통해 Bllossom-3B 모델로 답변 생성
    
    Args:
        prompt: build_prompt()에서 생성된 프롬프트
    
    Returns:
        str: 모델이 생성한 답변 텍스트
    """
    response = requests.post(
        'http://localhost:11434/api/generate',
        json={
            'model': 'cookie-chatbot-bllossom',
            'prompt': prompt,
            'stream': False,
            'options': {
                'temperature': 0.3,
                'top_p': 0.9,
                'num_predict': 256
            }
        }
    )
    
    return response.json()['response'].strip()

# 단독 테스트
test_prompt = build_prompt('초코퍼지 가격', filtered['filtered_results'])
answer = generate_answer(test_prompt)
print(f'모델 답변: {answer}')

모델 답변: 초코퍼지 3,500원이에요!


## 통합: RAG 파이프라인 함수

모듈 1~4를 하나로 연결한 최종 파이프라인입니다.

```
질문 → [모듈1] 검색 → [모듈2] 임계값 판단 → [모듈3] 프롬프트 조합 → [모듈4] 답변 생성
                              ↓ (임계값 초과 시)
                      "해당 정보를 찾을 수 없습니다"
```

In [51]:
def rag_pipeline(query: str, verbose: bool = False) -> str:
    """
    RAG 파이프라인 통합 함수
    
    사용자 질문 → 검색 → 임계값 판단 → 프롬프트 조합 → 답변 생성
    
    Args:
        query: 사용자 질문
        verbose: True이면 중간 과정(검색 결과, 거리 등)을 출력
    
    Returns:
        str: 최종 답변
    """
    # 모듈 1: ChromaDB 검색
    search_results = search_knowledge(query)
    
    if verbose:
        print(f'[검색] 질문: "{query}"')
        for r in search_results:
            print(f'  [{r["category"]}] {r["content"][:50]}... (거리: {r["distance"]:.4f})')
        print()
    
    # 모듈 2: 임계값 판단
    filter_result = filter_by_threshold(search_results)
    
    if verbose:
        print(f'[판단] 최소 거리: {filter_result["min_distance"]:.4f} '
              f'(임계값: {DISTANCE_THRESHOLD})')
        print(f'[판단] 관련 정보 존재: {filter_result["has_answer"]}')
        print()
    
    # 임계값 초과 → 정보 없음 응답 (1차 필터)
    if not filter_result['has_answer']:
        return '죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요.'
    
    # 모듈 3: 프롬프트 조합
    prompt = build_prompt(query, filter_result['filtered_results'])
    
    if verbose:
        print(f'[프롬프트] 참고 문서 {len(filter_result["filtered_results"])}개 포함')
        print()
    
    # 모듈 4: 답변 생성
    answer = generate_answer(prompt)
    
    return answer

## 파이프라인 테스트

다양한 유형의 질문으로 RAG 파이프라인의 동작을 검증합니다.

| 테스트 | 질문 유형 | 기대 동작 |
|--------|-----------|----------|
| 1 | 단일 정보 질문 | 정확한 가격 답변 |
| 2 | 영업 정보 질문 | 영업시간 답변 |
| 3 | 존재하지 않는 메뉴 | 임계값에서 차단 → 정보 없음 응답 |
| 4 | 위치 질문 | 주소/교통 정보 답변 |

In [52]:
# 테스트 질문 목록
test_questions = [
    '초코퍼지 얼마예요?',
    '오늘 몇 시까지 해요?',
    '마카롱 있어요?',
    '가게 어디에 있어요?',
    '일요일은 몇 시부터 몇 시까지 운영해요?',
    '일요일은 몇 시에 열어요?'
]

for q in test_questions:
    print(f'=' * 60)
    print(f'Q: {q}')
    print(f'-' * 60)
    answer = rag_pipeline(q, verbose=True)
    print(f'A: {answer}')
    print()

Q: 초코퍼지 얼마예요?
------------------------------------------------------------
[검색] 질문: "초코퍼지 얼마예요?"
  [메뉴] 초코퍼지 가격은 3,500원입니다.... (거리: 0.1766)
  [메뉴] 크랜베리&화이트초코칩 가격은 2,600원입니다.... (거리: 0.4493)
  [메뉴] 오렌지쿠키 가격은 2,600원입니다.... (거리: 0.4930)

[판단] 최소 거리: 0.1766 (임계값: 0.45)
[판단] 관련 정보 존재: True

[프롬프트] 참고 문서 2개 포함

A: 초코퍼지는 3,700원입니다!

Q: 오늘 몇 시까지 해요?
------------------------------------------------------------
[검색] 질문: "오늘 몇 시까지 해요?"
  [영업시간] 위니쿠키는 오후 5시에 마감합니다. 저녁이나 야간에는 운영하지 않습니다.... (거리: 0.4489)
  [영업시간] 위니쿠키 영업시간은 월요일부터 토요일까지 오전 10시 30분부터 오후 5시까지입니다.... (거리: 0.4706)
  [주문] 유통기한 관련 문의는 010-3387-5313으로 연락하시면 안내받을 수 있습니다.... (거리: 0.5472)

[판단] 최소 거리: 0.4489 (임계값: 0.45)
[판단] 관련 정보 존재: True

[프롬프트] 참고 문서 1개 포함

A: 오후 5시에 마감합니다!

Q: 마카롱 있어요?
------------------------------------------------------------
[검색] 질문: "마카롱 있어요?"
  [위치] 위니쿠키 앞쪽에 공용주차장이 있습니다.... (거리: 0.4922)
  [주문] 당일 준비된 쿠키는 매장 상황에 따라 달라질 수 있습니다. 방문 전 010-3387-531... (거리: 0.4926)
  [메뉴] 위니쿠키는 쿠키 전문점으로 쿠키와 휘낭시에만 판매합니다. 케이크, 마카롱, 빵, 음료 등은..

---

# 4단계: 환각 테스트

RAG 파이프라인이 다양한 유형의 질문에서 환각 없이 정확하게 동작하는지 체계적으로 검증합니다.

### 테스트 유형 (5가지)
| 유형 | 설명 | 환각 위험도 |
|------|------|------------|
| 1. 존재하지 않는 정보 | 메뉴에 없는 제품 질문 | ★★☆ |
| 2. 복합 질문 (있는 것 + 없는 것) | 정답이 섞여 있어 환각 유발 쉬움 | ★★★ |
| 3. 추론/비교 질문 | KB에 없는 판단을 요구 | ★★☆ |
| 4. 유사하지만 틀린 정보 | 존재하는 정보와 살짝 다른 질문 | ★★★ |
| 5. 범위 밖 질문 | 쿠키 관련이지만 답변 근거 없음 | ★★☆ |

In [53]:
# 환각 테스트 케이스 정의
hallucination_tests = [
    # 유형 1: 존재하지 않는 정보
    {
        'type': '1. 존재하지 않는 정보',
        'question': '마카롱 있어요?',
        'expected': '없다고 답하거나 정보 없음 응답'
    },
    {
        'type': '1. 존재하지 않는 정보',
        'question': '아메리카노 가격이 얼마예요?',
        'expected': '음료 메뉴가 없으므로 정보 없음 응답'
    },
    
    # 유형 2: 복합 질문 (가장 위험)
    {
        'type': '2. 복합 질문 (있는 것 + 없는 것)',
        'question': '초코퍼지랑 마카롱 가격 알려줘',
        'expected': '초코퍼지 가격만 답하고, 마카롱은 없다고 답해야 함'
    },
    {
        'type': '2. 복합 질문 (있는 것 + 없는 것)',
        'question': '르뱅쿠키 가격이랑 케이크 가격 알려줘',
        'expected': '르뱅쿠키 가격만 답하고, 케이크는 없다고 답해야 함'
    },
    
    # 유형 3: 추론/비교 질문
    {
        'type': '3. 추론/비교 질문',
        'question': '가장 인기 있는 메뉴가 뭐예요?',
        'expected': '인기 순위 정보가 없으므로 모른다고 답해야 함'
    },
    {
        'type': '3. 추론/비교 질문',
        'question': '제일 비싼 쿠키는 뭐예요?',
        'expected': '가격 비교 정보가 직접적으로 없으면 추측하지 말아야 함'
    },
    
    # 유형 4: 유사하지만 틀린 정보
    {
        'type': '4. 유사하지만 틀린 정보',
        'question': '위니쿠키 강남점 주소 알려줘',
        'expected': '강남점은 없으므로 정보 없음 응답 (기존 주소를 강남점이라고 하면 안 됨)'
    },
    {
        'type': '4. 유사하지만 틀린 정보',
        'question': '일요일에 몇 시에 열어요?',
        'expected': '일요일은 휴무이므로 정확히 휴무라고 답해야 함 (영업시간을 답하면 안 됨)'
    },
    
    # 유형 5: 범위 밖 질문
    {
        'type': '5. 범위 밖 질문',
        'question': '쿠키 만드는 레시피 알려줘',
        'expected': '레시피 정보가 없으므로 정보 없음 응답'
    },
    {
        'type': '5. 범위 밖 질문',
        'question': '사장님 이름이 뭐예요?',
        'expected': '사장 정보가 없으므로 정보 없음 응답'
    },
]

print(f'총 {len(hallucination_tests)}개 테스트 케이스 준비 완료')
for t in hallucination_tests:
    print(f'  [{t["type"]}] {t["question"]}')

총 10개 테스트 케이스 준비 완료
  [1. 존재하지 않는 정보] 마카롱 있어요?
  [1. 존재하지 않는 정보] 아메리카노 가격이 얼마예요?
  [2. 복합 질문 (있는 것 + 없는 것)] 초코퍼지랑 마카롱 가격 알려줘
  [2. 복합 질문 (있는 것 + 없는 것)] 르뱅쿠키 가격이랑 케이크 가격 알려줘
  [3. 추론/비교 질문] 가장 인기 있는 메뉴가 뭐예요?
  [3. 추론/비교 질문] 제일 비싼 쿠키는 뭐예요?
  [4. 유사하지만 틀린 정보] 위니쿠키 강남점 주소 알려줘
  [4. 유사하지만 틀린 정보] 일요일에 몇 시에 열어요?
  [5. 범위 밖 질문] 쿠키 만드는 레시피 알려줘
  [5. 범위 밖 질문] 사장님 이름이 뭐예요?


In [54]:
# 환각 테스트 실행
print('=' * 70)
print('  환각 테스트 시작 (RAG 파이프라인)')
print('=' * 70)
print()

test_results = []

for i, test in enumerate(hallucination_tests, 1):
    print(f'--- 테스트 {i}/{len(hallucination_tests)}: [{test["type"]}] ---')
    print(f'Q: {test["question"]}')
    print(f'기대: {test["expected"]}')
    print()
    
    # RAG 파이프라인 실행 (verbose로 중간 과정 확인)
    answer = rag_pipeline(test['question'], verbose=True)
    
    print(f'A: {answer}')
    print()
    
    # 결과 저장
    test_results.append({
        'type': test['type'],
        'question': test['question'],
        'expected': test['expected'],
        'answer': answer
    })
    
    print('=' * 70)
    print()

  환각 테스트 시작 (RAG 파이프라인)

--- 테스트 1/10: [1. 존재하지 않는 정보] ---
Q: 마카롱 있어요?
기대: 없다고 답하거나 정보 없음 응답

[검색] 질문: "마카롱 있어요?"
  [위치] 위니쿠키 앞쪽에 공용주차장이 있습니다.... (거리: 0.4922)
  [주문] 당일 준비된 쿠키는 매장 상황에 따라 달라질 수 있습니다. 방문 전 010-3387-531... (거리: 0.4926)
  [메뉴] 위니쿠키는 쿠키 전문점으로 쿠키와 휘낭시에만 판매합니다. 케이크, 마카롱, 빵, 음료 등은... (거리: 0.5109)

[판단] 최소 거리: 0.4922 (임계값: 0.45)
[판단] 관련 정보 존재: False

A: 죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요.


--- 테스트 2/10: [1. 존재하지 않는 정보] ---
Q: 아메리카노 가격이 얼마예요?
기대: 음료 메뉴가 없으므로 정보 없음 응답

[검색] 질문: "아메리카노 가격이 얼마예요?"
  [메뉴] 아망디오쇼콜라 가격은 2,600원입니다.... (거리: 0.4441)
  [메뉴] 오렌지쿠키 가격은 2,600원입니다.... (거리: 0.5000)
  [메뉴] 메이플피칸쿠키 가격은 2,600원입니다.... (거리: 0.5086)

[판단] 최소 거리: 0.4441 (임계값: 0.45)
[판단] 관련 정보 존재: True

[프롬프트] 참고 문서 1개 포함

A: 죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요!


--- 테스트 3/10: [2. 복합 질문 (있는 것 + 없는 것)] ---
Q: 초코퍼지랑 마카롱 가격 알려줘
기대: 초코퍼지 가격만 답하고, 마카롱은 없다고 답해야 함

[검색] 질문: "초코퍼지랑 마카롱 가격 알려줘"
  [메뉴] 초코퍼지 가격은 3,500원입니다.... (거리: 0.2512)
  [메뉴] 크랜베리&화이트초코칩 가격은 2,600원입니다.... (거리: 0.38

## 환각 테스트 결과 요약

각 테스트 결과를 확인하고, 환각 발생 여부를 수동으로 판정합니다.

### 판정 기준
- **PASS**: 기대한 대로 정확하게 답변하거나, 모르는 건 모른다고 답함
- **FAIL**: 없는 정보를 만들어내거나, 틀린 정보를 답함
- **PARTIAL**: 일부는 맞지만 일부에서 환각 발생

In [55]:
# 결과 요약 출력
print('=' * 70)
print('  환각 테스트 결과 요약')
print('=' * 70)
print()

for i, r in enumerate(test_results, 1):
    print(f'{i:2d}. [{r["type"]}]')
    print(f'    Q: {r["question"]}')
    print(f'    A: {r["answer"]}')
    print(f'    기대: {r["expected"]}')
    print(f'    판정: ___  ← PASS / FAIL / PARTIAL 직접 기입')
    print()

  환각 테스트 결과 요약

 1. [1. 존재하지 않는 정보]
    Q: 마카롱 있어요?
    A: 죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요.
    기대: 없다고 답하거나 정보 없음 응답
    판정: ___  ← PASS / FAIL / PARTIAL 직접 기입

 2. [1. 존재하지 않는 정보]
    Q: 아메리카노 가격이 얼마예요?
    A: 죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요!
    기대: 음료 메뉴가 없으므로 정보 없음 응답
    판정: ___  ← PASS / FAIL / PARTIAL 직접 기입

 3. [2. 복합 질문 (있는 것 + 없는 것)]
    Q: 초코퍼지랑 마카롱 가격 알려줘
    A: 죄송합니다, 해당 정보를 찾을 수 없습니다. 010-3387-5313으로 문의해 주세요!
    기대: 초코퍼지 가격만 답하고, 마카롱은 없다고 답해야 함
    판정: ___  ← PASS / FAIL / PARTIAL 직접 기입

 4. [2. 복합 질문 (있는 것 + 없는 것)]
    Q: 르뱅쿠키 가격이랑 케이크 가격 알려줘
    A: 르뱅쿠키는 4,300원이에요!
    기대: 르뱅쿠키 가격만 답하고, 케이크는 없다고 답해야 함
    판정: ___  ← PASS / FAIL / PARTIAL 직접 기입

 5. [3. 추론/비교 질문]
    Q: 가장 인기 있는 메뉴가 뭐예요?
    A: 르뱅쿠키, 비스퀴, 초코퍼지!
    기대: 인기 순위 정보가 없으므로 모른다고 답해야 함
    판정: ___  ← PASS / FAIL / PARTIAL 직접 기입

 6. [3. 추론/비교 질문]
    Q: 제일 비싼 쿠키는 뭐예요?
    A: 르뱅쿠키(4,300원)입니다!
    기대: 가격 비교 정보가 직접적으로 없으면 추측하지 말아야 함
    판정: ___  ← PASS / FAIL / PARTIAL 직접 기입

 7.